# 05 — The numbers

Five things to measure, each with error bars.

**With 40 cases, "88%" is misleading** — it sounds precise and it isn't. Report
`0.88 [0.74, 0.96]` instead.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

runs = {p.stem: json.load(open(p)) for p in sorted(Path("data/results").glob("*.json"))}
sorted(runs)


## Error bars

Wilson for proportions (correct at small n, unlike the usual formula). Bootstrap for F1 —
resample the cases 1000 times and look at the spread.

In [ ]:
import math

def wilson(successes, n, z=1.96):
    """95% interval for a proportion."""
    if n == 0:
        return (float("nan"),) * 3
    p = successes / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return p, max(0, centre - half), min(1, centre + half)

def bootstrap(items, stat, n_resamples=1000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(items)
    draws = [stat([items[i] for i in rng.integers(0, n, n)]) for _ in range(n_resamples)]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return stat(items), float(lo), float(hi)

def fmt(point, lo, hi):
    return f"{point:.2f} [{lo:.2f}, {hi:.2f}]"

print(fmt(*wilson(35, 40)))


## The five measurements

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

LABELS = ["met", "unmet", "insufficient_evidence"]

def macro_f1(rows):
    return f1_score([r["gold"] for r in rows], [r["pred"] for r in rows],
                    labels=LABELS, average="macro", zero_division=0)

def summarise(run):
    rows = run["rows"]
    n = len(rows)
    found   = wilson(sum(r.get("gold_chunk_retrieved", False) for r in rows), n)
    f1      = bootstrap(rows, macro_f1)
    exact   = wilson(sum(r.get("status") == "exact" for r in rows), n)
    made_up = wilson(sum(r.get("status") == "made_up" for r in rows), n)
    absts   = wilson(sum(r["pred"] == "insufficient_evidence" for r in rows), n)
    answered = [r for r in rows if r["pred"] != "insufficient_evidence"]
    acc      = wilson(sum(r["gold"] == r["pred"] for r in answered), len(answered))
    return [fmt(*found), fmt(*f1), fmt(*exact), fmt(*made_up), fmt(*absts), fmt(*acc)]


## The table

In [ ]:
ORDER = ["row0_context_only", "row1_naive", "row2_structure",
         "row3_hybrid", "row4_rerank", "row5_full"]
HEAD = ["config", "found rule", "F1", "quotes real", "made up", "abstained", "acc. answered"]

print(" | ".join(HEAD))
for name in ORDER:
    if name in runs:
        print(" | ".join([name] + summarise(runs[name])))

# then paste this into README.md


## Confusion matrix

In [ ]:
rows = runs["row5_full"]["rows"]
cm = confusion_matrix([r["gold"] for r in rows], [r["pred"] for r in rows], labels=LABELS)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_xticks(range(3), LABELS, rotation=45, ha="right")
ax.set_yticks(range(3), LABELS)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout()


## Risk-coverage curve

Sweep a confidence threshold: how many questions it answers vs how wrong it is on the ones it
answered. The shape of this curve is the entire argument for letting it abstain.

In [ ]:
def risk_coverage(rows):
    scored = sorted(rows, key=lambda r: -r.get("score", 0))
    correct = np.array([r["gold"] == r["pred"] for r in scored], dtype=float)
    cov = np.arange(1, len(scored) + 1) / len(scored)
    err = 1 - np.cumsum(correct) / np.arange(1, len(scored) + 1)
    return cov, err

cov, err = risk_coverage(runs["row5_full"]["rows"])
plt.plot(cov, err)
plt.xlabel("coverage (fraction answered)")
plt.ylabel("error rate on answered")
plt.tight_layout()


## Handwritten vs templated

If the model does much worse on the 10 I wrote by hand, my templates were too easy and the
README has to say so.

In [ ]:
cases = {c["id"]: c for c in json.load(open("data/cases.json"))}
rows  = runs["row5_full"]["rows"]
hand  = [r for r in rows if cases[r["case_id"]]["hand_written"]]
tmpl  = [r for r in rows if not cases[r["case_id"]]["hand_written"]]
print("handwritten:", fmt(*bootstrap(hand, macro_f1)))
print("templated:  ", fmt(*bootstrap(tmpl, macro_f1)))
